# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Maullana Ilham Ar Rosyiid Mariandi  
**NIM:** 202310370311093  
**Username GitHub:** maullanailhammm  

# Milestone 3: EDA dan Insights

## Bagian 1: Inisialisasi dan Memuat Data Bersih

Analisis temporal artikel berita Indonesia menggunakan data hasil cleaning dari Milestone 2. Dataset tidak memiliki koordinat geografis, sehingga analisis difokuskan pada pola waktu dan perbandingan antar-sumber.

Analisis ini menggunakan Polars untuk transformasi data dan Plotly untuk visualisasi interaktif.

Data yang digunakan adalah hasil cleaning dari Milestone 2 pada `data/processed/dataset_bersih.parquet`. Notebook mencari root project secara otomatis apabila dijalankan dari folder `notebooks`, lalu membaca file Parquet dengan `pl.read_parquet()`.

Kolom `date` diparse dari teks ke `Datetime` dan disimpan sebagai `tanggal_waktu`. Kolom `title` juga diubah menjadi `panjang_judul_eda` untuk analisis hubungan volume publikasi dengan panjang judul. Sel kode menampilkan jumlah baris, rentang waktu, dan jumlah sumber berita.

In [1]:
from pathlib import Path

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data bersih tidak ditemukan: {DATA_PATH}")

df = pl.read_parquet(DATA_PATH).with_columns([
    pl.col("date").str.slice(0, 19).str.strptime(
        pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=True
    ).alias("tanggal_waktu"),
    pl.col("title").str.len_chars().alias("panjang_judul_eda"),
])
print(f"Data loaded: {df.height:,} baris x {df.width} kolom")
print(f"Periode: {df['tanggal_waktu'].min()} sampai {df['tanggal_waktu'].max()}")
print(f"Sumber berita: {df['source'].n_unique()}")

Data loaded: 32,725 baris x 15 kolom
Periode: 2018-08-10 05:11:58 sampai 2023-04-11 20:13:25
Sumber berita: 7


## Bagian 2: Analisis Temporal dan Window Function


Agregasi harian memakai `group_by_dynamic()` dengan interval satu hari. Setelah jumlah artikel per hari dihitung, `rolling_mean(window_size=7, min_samples=1)` menghasilkan rata-rata bergerak tujuh hari untuk melihat tren yang lebih halus.

Selain agregasi harian, notebook membentuk ringkasan bulanan total (`bulanan`) dan ringkasan bulanan berdasarkan sumber berita (`bulanan_sumber`). Tabel `pola_waktu` digunakan untuk menghitung pola publikasi menurut hari dalam minggu dan jam, sedangkan `volume_judul` membandingkan volume bulanan dengan rata-rata panjang judul.

In [2]:
harian = (
    df.sort("tanggal_waktu")
    .group_by_dynamic("tanggal_waktu", every="1d")
    .agg(pl.len().alias("jumlah_artikel"))
    .sort("tanggal_waktu")
    .with_columns(
        pl.col("jumlah_artikel").rolling_mean(window_size=7, min_samples=1)
        .alias("rata_rata_bergerak_7_hari")
    )
)
bulanan = (
    df.with_columns(pl.col("tanggal_waktu").dt.truncate("1mo").alias("bulan"))
    .group_by("bulan").agg(pl.len().alias("jumlah_artikel")).sort("bulan")
)
bulanan_sumber = (
    df.with_columns(pl.col("tanggal_waktu").dt.truncate("1mo").alias("bulan"))
    .group_by(["bulan", "source"]).agg(pl.len().alias("jumlah_artikel"))
    .sort(["bulan", "source"])
)
sumber = df.group_by("source").agg(pl.len().alias("jumlah_artikel")).sort("jumlah_artikel", descending=True)
waktu = df.with_columns([
    pl.col("tanggal_waktu").dt.strftime("%A").alias("hari_minggu"),
    pl.col("tanggal_waktu").dt.hour().alias("jam"),
])
urutan_hari = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pola_waktu = waktu.group_by(["hari_minggu", "jam"]).agg(pl.len().alias("jumlah_artikel"))
volume_judul = (
    df.with_columns(pl.col("tanggal_waktu").dt.truncate("1mo").alias("bulan"))
    .group_by("bulan")
    .agg([
        pl.len().alias("jumlah_artikel"),
        pl.col("panjang_judul_eda").mean().alias("rata_rata_panjang_judul"),
    ])
    .sort("bulan")
)
harian.head()

tanggal_waktu,jumlah_artikel,rata_rata_bergerak_7_hari
datetime[μs],u32,f64
2018-08-10 00:00:00,1,1.0
2023-03-04 00:00:00,37,19.0
2023-03-05 00:00:00,41,26.333333
2023-03-06 00:00:00,54,33.25
2023-03-07 00:00:00,76,41.8


## Bagian 3: Enam Visualisasi Interaktif (Plotly)

Visualisasi berikut disesuaikan dengan dataset artikel berita yang tidak memiliki koordinat geografis. Analisis difokuskan pada pola temporal, perbandingan sumber berita, dan hubungan antar-variabel agregat:

- **Visualisasi 1 - Tren publikasi harian:** line chart jumlah artikel per hari dengan rata-rata bergerak 7 hari.
- **Visualisasi 2 - Distribusi volume harian:** histogram jumlah artikel yang dipublikasikan per hari.
- **Visualisasi 3 - Komposisi sumber berita:** bar chart jumlah artikel dari setiap sumber.
- **Visualisasi 4 - Tren bulanan per sumber:** line chart untuk membandingkan perubahan volume antar-sumber berita.
- **Visualisasi 5 - Pola hari dan jam:** heatmap jumlah artikel berdasarkan hari dalam minggu dan jam publikasi.
- **Visualisasi 6 - Volume dan panjang judul:** scatter plot hubungan jumlah artikel bulanan dengan rata-rata panjang judul.

Seluruh grafik dibuat sebagai visualisasi interaktif menggunakan Plotly.

In [3]:
fig1 = px.line(harian.to_pandas(), x="tanggal_waktu", y=["jumlah_artikel", "rata_rata_bergerak_7_hari"], title="Tren Publikasi Harian dan Rata-rata Bergerak 7 Hari")
fig1.update_layout(hovermode="x unified")
fig1.show()

In [4]:
fig2 = px.histogram(harian.to_pandas(), x="jumlah_artikel", nbins=40, title="Distribusi Jumlah Artikel per Hari")
fig2.show()

In [5]:
fig3 = px.bar(sumber.to_pandas(), x="source", y="jumlah_artikel", color="source", title="Jumlah Artikel menurut Sumber Berita")
fig3.show()

In [6]:
fig4 = px.line(bulanan_sumber.to_pandas(), x="bulan", y="jumlah_artikel", color="source", title="Tren Bulanan Artikel per Sumber")
fig4.show()

In [7]:
fig5 = px.density_heatmap(
    pola_waktu.to_pandas(), x="jam", y="hari_minggu", z="jumlah_artikel",
    category_orders={"hari_minggu": urutan_hari},
    title="Pola Publikasi menurut Hari dan Jam",
)
fig5.show()

In [8]:
fig6 = px.scatter(
    volume_judul.to_pandas(), x="jumlah_artikel", y="rata_rata_panjang_judul",
    size="jumlah_artikel", hover_data=["bulan"],
    title="Volume Bulanan dan Rata-rata Panjang Judul",
)
fig6.show()

## Lima insight analitik utama

Insight berikut dihitung dari tabel agregasi yang digunakan oleh visualisasi.

In [9]:
sumber_terbanyak = sumber.row(0, named=True)
hari_tertinggi = harian.sort("jumlah_artikel", descending=True).row(0, named=True)
bulan_tertinggi = bulanan.sort("jumlah_artikel", descending=True).row(0, named=True)
bulan_terendah = bulanan.sort("jumlah_artikel").row(0, named=True)
puncak_waktu = pola_waktu.sort("jumlah_artikel", descending=True).row(0, named=True)
korelasi = volume_judul.select(pl.corr("jumlah_artikel", "rata_rata_panjang_judul")).item()
insights = [
    f"Sumber terbanyak: {sumber_terbanyak['source']} ({sumber_terbanyak['jumlah_artikel']:,} artikel).",
    f"Hari dengan publikasi terbanyak: {hari_tertinggi['tanggal_waktu']} ({hari_tertinggi['jumlah_artikel']:,} artikel).",
    f"Bulan tertinggi: {bulan_tertinggi['bulan']} ({bulan_tertinggi['jumlah_artikel']:,}); bulan terendah: {bulan_terendah['bulan']} ({bulan_terendah['jumlah_artikel']:,}).",
    f"Puncak hari-jam: {puncak_waktu['hari_minggu']} pukul {puncak_waktu['jam']:02d}:00 UTC ({puncak_waktu['jumlah_artikel']:,} artikel).",
    f"Korelasi volume bulanan dengan panjang judul rata-rata: {korelasi:.3f}; hubungan perlu dibaca bersama scatter plot.",
]
for nomor, insight in enumerate(insights, start=1):
    print(f"{nomor}. {insight}")

1. Sumber terbanyak: kumparan (8,579 artikel).
2. Hari dengan publikasi terbanyak: 2023-03-16 00:00:00 (1,583 artikel).
3. Bulan tertinggi: 2023-03-01 00:00:00 (21,213); bulan terendah: 2018-08-01 00:00:00 (1).
4. Puncak hari-jam: Tuesday pukul 10:00 UTC (586 artikel).
5. Korelasi volume bulanan dengan panjang judul rata-rata: 0.874; hubungan perlu dibaca bersama scatter plot.
